In [ ]:
import joblib
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Conv1D, MaxPooling1D, Flatten, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from google.colab import drive
import matplotlib.pyplot as plt

# 1. Kết nối Drive
try:
    drive.mount('/content/drive')
except:
    pass

# 2. Đường dẫn thư mục
save_dir = "/content/drive/MyDrive/DoAn_NIDS/Dataset/"

# 3. Load dữ liệu (File FINAL đã Shuffle ở bước DNN)
print("⏳ Đang load dữ liệu từ Drive...")
try:
    X_train = joblib.load(save_dir + 'X_train_final.pkl')
    y_train = joblib.load(save_dir + 'y_train_final.pkl')
    X_test = joblib.load(save_dir + 'X_test_final.pkl')
    y_test = joblib.load(save_dir + 'y_test_final.pkl') # Load y_test để lát tính số lớp
    print(f"✅ Đã load xong: {len(X_train)} mẫu train.")
except Exception as e:
    print(f"❌ Lỗi load dữ liệu: {e}")
    print("Anh hãy kiểm tra lại xem đã chạy bước xử lý dữ liệu (Kịch bản 1) chưa nhé!")

In [ ]:
# --- RESHAPE DỮ LIỆU ---
print("-" * 40)
print("🛠️ Đang Reshape dữ liệu sang dạng 3D cho CNN...")

# Input CNN phải là (Samples, TimeSteps, Features)
# Với dữ liệu bảng, TimeSteps = Số cột, Features = 1

# 1. Reshape tập Train
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)

# 2. Reshape tập Test (để lát đánh giá)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# 3. Lấy thông số Input Shape
input_shape = (X_train_cnn.shape[1], 1)
n_classes = len(np.unique(y_train))

print(f"   - Shape cũ (DNN): {X_train.shape}")
print(f"   - Shape mới (CNN): {X_train_cnn.shape}")
print(f"   - Input Shape: {input_shape}")
print(f"   - Số lớp đầu ra: {n_classes}")

In [ ]:
def build_cnn_model_v2():
    model = Sequential(name="CNN_Model_Scenario_2_V2")

    # --- Block 1: Tầm nhìn rộng (Wide Vision) ---
    # Kernel=7: Nhìn 7 cột cùng lúc (giảm nhiễu cục bộ)
    # Padding='same': Giữ nguyên kích thước dữ liệu sau khi quét
    model.add(Conv1D(filters=128, kernel_size=7, padding='same', input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling1D(pool_size=2))

    # --- Block 2: Tầm nhìn trung bình (Deep Features) ---
    # Kernel=5: Nhìn 5 cột
    # Filters=256: Tăng gấp đôi thám tử
    model.add(Conv1D(filters=256, kernel_size=5, padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.25)) # Chống học vẹt nhẹ

    # --- Block 3: Tầm nhìn chi tiết (Fine Features) ---
    # Kernel=3: Nhìn 3 cột
    # Filters=512: Rất nhiều thám tử để chốt đặc trưng cuối
    model.add(Conv1D(filters=512, kernel_size=3, padding='same'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    # --- Chuyển đổi ---
    model.add(Flatten())

    # --- Phân loại ---
    # Lớp Dense lớn (256) để xử lý lượng thông tin khổng lồ từ CNN
    model.add(Dense(256, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dropout(0.4)) # Dropout mạnh hơn ở lớp cuối

    # Output Layer
    model.add(Dense(n_classes, activation='softmax'))

    # Compile với Learning Rate NHỎ (0.0001) để biểu đồ mượt, không bị gai (spikes)
    optimizer = Adam(learning_rate=0.0001)
    model.compile(loss='sparse_categorical_crossentropy',
                  optimizer=optimizer,
                  metrics=['accuracy'])

    return model

print("🏗️ Đang khởi tạo kiến trúc CNN V2...")
model_cnn_v2 = build_cnn_model_v2()
model_cnn_v2.summary()

In [ ]:
# --- HUẤN LUYỆN ---
print("-" * 40)
print("🚀 Bắt đầu huấn luyện CNN V2 (Chậm mà chắc)...")

# Đường dẫn lưu model
model_path_v2 = save_dir + 'Scenario2_CNN_V2_Best.keras'

callbacks_v2 = [
    # Lưu phiên bản tốt nhất
    ModelCheckpoint(model_path_v2, monitor='val_loss', save_best_only=True, mode='min', verbose=1),

    # Kiên nhẫn 8 vòng (vì model này học chậm nên cần chờ lâu hơn chút)
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),

    # Giảm tốc khi gặp khó
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# Thực thi
history_v2 = model_cnn_v2.fit(
    X_train_cnn, y_train,
    validation_split=0.1,
    epochs=60,            # Số vòng tối đa
    batch_size=1024,      # Batch lớn giúp chạy ổn định
    callbacks=callbacks_v2,
    verbose=1
)

print("\n✅ HUẤN LUYỆN HOÀN TẤT!")

# --- VẼ BIỂU ĐỒ NGAY ---
plt.figure(figsize=(14, 5))

# Biểu đồ Loss
plt.subplot(1, 2, 1)
plt.plot(history_v2.history['loss'], label='Train Loss', color='blue')
plt.plot(history_v2.history['val_loss'], label='Val Loss', color='orange')
plt.title('Biểu đồ Loss (V2)')
plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)

# Biểu đồ Accuracy
plt.subplot(1, 2, 2)
plt.plot(history_v2.history['accuracy'], label='Train Acc', color='green')
plt.plot(history_v2.history['val_accuracy'], label='Val Acc', color='red')
plt.title('Biểu đồ Accuracy (V2)')
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.legend(); plt.grid(True)

plt.show()